
# Slone & Netzer 2012 disc: Black-hole mass and Eddington ratio

The Slone & Netzer (2012) accretion-disc library (SN12, as packaged by
AGNfitter-rX) tabulates the big-blue-bump continuum over black-hole mass and
Eddington ratio. The disc's characteristic temperature scales as
$T_\mathrm{max} \propto (\dot m / M_\mathrm{BH})^{1/4}$, so the
spectral peak walks across the UV/optical as those two knobs change:

- **More massive black holes** have larger, cooler discs — the big blue bump
  shifts redward (toward the optical).

- **Higher Eddington ratios** push more material through the inner disc —
  the peak moves blueward and the disc brightens.

The two panels sweep each parameter in turn (the other held fixed), with the
swept value encoded by color. Bolometric luminosity is held fixed across each
sweep so the curves isolate the *spectral shape* — the SN12 template shape is
governed purely by $(M_\mathrm{BH}, \dot m)$, with $L_\mathrm{bol}$
setting only the normalization — and the peak migration stands out cleanly.

The SN12 grid is reproduced node-exactly in tengri via monotone-cubic
interpolation, so these curves track the AGNfitter templates at the tabulated
nodes.

## References
.. [1] A. Slone & H. Netzer, "The effect of disc winds on the structure and
   spectrum of accretion discs," MNRAS 426, 656 (2012). arXiv:1207.5077.
   https://doi.org/10.1111/j.1365–2966.2012.21699.x
.. [2] L. N. Martínez-Ramírez et al., "AGNfitter-rx: Modeling the radio-to-X-ray
   spectral energy distributions of AGNs," A&A 688, A46 (2024).
   arXiv:2405.12111. https://doi.org/10.1051/0004–6361/202449329


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style
from tengri.utils.physics_constants import C_AA  # speed of light [Angstrom/s]

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")
warnings.filterwarnings("ignore", message=".*deprecated.*")

ssp = tengri.load_ssp()

# Minimal host: stellar light suppressed so the disc continuum stands alone.
SFH = {"type": "const", "all_params": tengri.FIXED, "log_total_mass": -10.0}
DUST = {"type": "two_component", "all_params": tengri.FIXED, "tau_diff": 0.0, "tau_bc": 0.0}

# SN12 grid extent: log_mbh in [7.4, 9.8], log_ledd in [-4.0, -1.96]. Stay inside.
# Mass sweep starts at 8.0 so every disc peaks inside the grid (the bluest grid
# node is 451 AA; lower masses peak at/below it and clip).
LOG_LEDD_REF = -2.5  # fixed Eddington ratio for the mass panel
LOG_MBH_REF = 8.5  # fixed black-hole mass for the Eddington panel
LOG_MBH_VALUES = np.linspace(8.0, 9.7, 7)
LOG_LEDD_VALUES = np.linspace(-3.8, -2.0, 7)

# L_bol held fixed [log10(L_bol / L_sun)] so the panels isolate spectral shape.
LOG_LBOL_FIXED = 11.0


def disc_sed(log_mbh: float, log_ledd: float) -> tuple[np.ndarray, np.ndarray]:
    """Return (wavelength [AA], nu*L_nu [erg/s]) for one SN12 disc state."""
    log_lbol = LOG_LBOL_FIXED  # shape set by (M_BH, Edd); L_bol only normalizes
    model = tengri.SEDModel.build(
        ssp,
        sfh=SFH,
        dust=DUST,
        agn={
            "disc": {"type": "slone_netzer", "all_params": tengri.FIXED},
            "all_params": tengri.FIXED,
            "log_lbol": log_lbol,
            "log_mbh": log_mbh,
            "log_ledd": log_ledd,
            "lum_ratio": 1.0,
        },
        redshift=tengri.Fixed(0.05),
    )
    p = dict(model.spec.sample(jax.random.PRNGKey(0)))
    out = model.predict(p)
    wave = np.asarray(model.wavelengths)
    return wave, C_AA / wave * np.asarray(out.rest_sed())


fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.6), sharey=True)

# ── Panel 1: black-hole mass sweep (fixed Eddington ratio) ──────────────
ax = axes[0]
norm_m = mpl.colors.Normalize(vmin=LOG_MBH_VALUES.min(), vmax=LOG_MBH_VALUES.max())
cmap_m = plt.get_cmap("viridis")
for log_mbh in LOG_MBH_VALUES:
    wave, nu_l_nu = disc_sed(log_mbh, LOG_LEDD_REF)
    ax.loglog(wave, nu_l_nu, color=cmap_m(norm_m(log_mbh)), lw=1.6)
ax.set_title(rf"Mass sweep  ($\log \lambda_\mathrm{{Edd}} = {LOG_LEDD_REF:.1f}$)", fontsize=10)
ax.set_ylabel(r"$\nu L_\nu$  [erg s$^{-1}$]")
sm_m = mpl.cm.ScalarMappable(norm=norm_m, cmap=cmap_m)
cb_m = fig.colorbar(sm_m, ax=ax, pad=0.01)
cb_m.set_label(r"$\log_{10}(M_\mathrm{BH} / M_\odot)$", fontsize=9)

# ── Panel 2: Eddington-ratio sweep (fixed black-hole mass) ──────────────
ax = axes[1]
norm_e = mpl.colors.Normalize(vmin=LOG_LEDD_VALUES.min(), vmax=LOG_LEDD_VALUES.max())
cmap_e = plt.get_cmap("cividis")
for log_ledd in LOG_LEDD_VALUES:
    wave, nu_l_nu = disc_sed(LOG_MBH_REF, log_ledd)
    ax.loglog(wave, nu_l_nu, color=cmap_e(norm_e(log_ledd)), lw=1.6)
ax.set_title(rf"Eddington sweep  ($\log M_\mathrm{{BH}} = {LOG_MBH_REF:.1f}$)", fontsize=10)
sm_e = mpl.cm.ScalarMappable(norm=norm_e, cmap=cmap_e)
cb_e = fig.colorbar(sm_e, ax=ax, pad=0.01)
cb_e.set_label(r"$\log_{10}(\dot m)$ [Eddington ratio]", fontsize=9)

for ax in axes:
    ax.set_xlim(5e2, 3e4)
    ax.set_ylim(9e43, 3.2e44)
    ax.set_xlabel(r"Rest-frame wavelength $\lambda$  [$\mathrm{\AA}$]")
    ax.grid(True, which="major", alpha=0.2)
    ax.axvspan(3000, 8000, color="0.82", alpha=0.4, zorder=0)  # optical band
    ax.text(4900, 9.8e43, "optical", fontsize=8, ha="center", color="0.4", style="italic")

fig.suptitle(
    "Slone & Netzer 2012 disc: black-hole mass vs Eddington ratio",
    fontsize=11.5,
    weight="bold",
)
fig.tight_layout()
plt.savefig("plot_slone_netzer_disc_sweep.png", dpi=150, bbox_inches="tight")